In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
match=pd.read_csv("matches.csv")
delivery=pd.read_csv("deliveries.csv")
match.head()

In [ ]:
match.dtypes

In [ ]:
match.shape

In [ ]:
delivery.head()

In [ ]:
delivery.dtypes

- batting_team  
- bowling team
- city
- runs_left
- balls_left
- wicket_left 
- total_runs_x 
- crr 
- rrr
- result


In [ ]:
total_score_df=delivery.groupby(['match_id','inning']).sum()['total_runs'].reset_index()

In [ ]:
total_score_df

In [ ]:
total_score_df=total_score_df[total_score_df['inning']==1]

In [ ]:
match_df=match.merge(total_score_df[['match_id','total_runs']],left_on='id',right_on='match_id')

In [ ]:
match_df

In [ ]:
match_df['team1'].unique()

In [ ]:
teams=[
    'Sunrisers Hyderabad',
    'Mumbai Indians', 
    'Royal Challengers Bangalore',
    'Kolkata Knight Riders', 
    'Kings XI Punjab',
    'Chennai Super Kings', 
    'Rajasthan Royals', 
    'Delhi Capitals'
]

In [ ]:
match_df['team1']=match_df['team1'].str.replace("Delhi Daredevils","Delhi Capitals")
match_df['team2']=match_df['team2'].str.replace("Delhi Daredevils","Delhi Capitals")

match_df['team1']=match_df['team1'].str.replace("Deccan Chargers","Sunrisers Hyderabad")
match_df['team2']=match_df['team2'].str.replace("Deccan Chargers","Sunrisers Hyderabad")



In [ ]:
match_df=match_df[match_df['team1'].isin(teams)]
match_df=match_df[match_df['team2'].isin(teams)]

In [ ]:
match_df.shape

In [ ]:
match_df.head()

In [ ]:
match_df.shape

In [ ]:
match_df=match_df[match_df['dl_applied']==0]

In [ ]:
match_df=match_df[['match_id','city','winner','total_runs']]

In [ ]:
delivery_df=match_df.merge(delivery,on='match_id')

In [ ]:
delivery_df=delivery_df[delivery_df['inning']==2]

In [ ]:
delivery_df.shape

In [ ]:
delivery_df.head()

In [ ]:
delivery_df.shape

In [ ]:
delivery_df['current_score']=delivery_df.groupby('match_id')['total_runs_y'].cumsum()

In [ ]:
delivery_df

In [ ]:
delivery_df['runs_left']=delivery_df['total_runs_x']-delivery_df['current_score']

In [ ]:
delivery_df['ball_left']=126-(delivery_df['over']*6+delivery_df['ball'])

In [ ]:
delivery_df['player_dismissed']=delivery_df['player_dismissed'].fillna('0')
delivery_df['player_dismissed']=delivery_df['player_dismissed'].apply(lambda x:x if x=='0' else '1')
delivery_df['player_dismissed']=delivery_df['player_dismissed'].astype('int')
wickets=delivery_df.groupby('match_id')['player_dismissed'].cumsum().values
delivery_df['wickets_left']=10-wickets
delivery_df.head()

In [ ]:
delivery_df.tail()

In [ ]:
##crr
delivery_df['crr']=(delivery_df['current_score']*6)/(120-delivery_df['ball_left'])

In [ ]:
delivery_df['rrr']=(delivery_df['runs_left']*6)/delivery_df['ball_left']

In [ ]:
def result(row):
    return 1 if row['batting_team']==row['winner'] else 0


In [ ]:
delivery_df['result']=delivery_df.apply(result,axis=1)

In [ ]:
final_df=delivery_df[['batting_team','bowling_team','city','runs_left','ball_left','wickets_left','total_runs_x','current_score','crr','rrr','result']]

In [ ]:
final_df=final_df.sample(final_df.shape[0])

In [ ]:
final_df.sample()

In [ ]:
final_df.isnull().sum()

In [ ]:
final_df=final_df[final_df['ball_left']!=0]

In [ ]:
final_df.dropna(inplace=True)

In [ ]:
team_wins=final_df.groupby('batting_team')['result'].mean()*100
team_wins=team_wins.sort_values(ascending=False)
plt.figure(figsize=(12,6))
sns.barplot(x=team_wins.index,y=team_wins.values,palette='viridis')
plt.xticks(rotation=45)
plt.title("Team Wise Win Percentage")

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(final_df['crr'],final_df['rrr'],cmap='coolwarm',alpha=0.3)
plt.colorbar(label="0 = lose,1 = win")
plt.xlabel('crr')
plt.ylabel('rrr')
plt.title("CRR vs RRR = Win/Loss Pattern")
plt.show()


In [ ]:
final_df.shape

In [ ]:
X=final_df.iloc[:,:-1]
y=final_df.iloc[:,-1]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
trf=ColumnTransformer([
    ('trf',OneHotEncoder(sparse_output=False,drop='first'),['batting_team','bowling_team','city'])
],
remainder='passthrough')

In [ ]:
from sklearn.model_selection import GridSearchCV
X_train_tr=trf.fit_transform(X_train)
X_test_tr=trf.transform(X_test)
params={
    'max_iter':[2000],
    'solver':['liblinear',],
    'C':[0.1,1,10]
}
gs=GridSearchCV(LogisticRegression(),params,cv=3,scoring='accuracy',n_jobs=1)
gs.fit(X_train_tr,y_train)
print("best params :",gs.best_params_)
print("Best Accuracy :",gs.best_score_)


In [ ]:
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score
final_model=LogisticRegression(C=10,max_iter=2000,solver='liblinear')
final_model.fit(X_train_tr,y_train)
y_pred=final_model.predict(X_test_tr)
print("Test Accuracy :",accuracy_score(y_test,y_pred))
print("F1 Score :",f1_score(y_test,y_pred))
print("Recall Score :",recall_score(y_test,y_pred))
print("Precision Score :",precision_score(y_test,y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(X_train_tr,y_train)
y_pred_rf=rf.predict(X_test_tr)
print("random forest :",accuracy_score(y_test,y_pred_rf))

In [ ]:
from sklearn.model_selection import GridSearchCV
params_rf={
    'n_estimators':[50,100,200],
    'max_depth':[5,10,15],
    'min_samples_split':[5,10,20]
}
gs_rf=GridSearchCV(RandomForestClassifier(random_state=42),params_rf,cv=3,scoring='accuracy',n_jobs=1)
gs_rf.fit(X_train_tr,y_train)
print('Best_params :',gs_rf.best_params_)
print("Best Accuracy :",gs_rf.best_score_)

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np
best_rf=gs_rf.best_estimator_
train_acc=accuracy_score(y_train,best_rf.predict(X_train_tr))
cv_cores=cross_val_score(best_rf,X_train_tr,y_train,cv=10,scoring='accuracy')
print("Train Accuracy :",train_acc)
print("Mean accuracy :",np.mean(cv_cores))
print("cv std :",np.std(cv_cores))
print("Test_acc:",accuracy_score(y_test,best_rf.predict(X_test_tr)))

In [ ]:
import pickle
pickle.dump(best_rf,open('ipl_model.pkl','wb'))
pickle.dump(trf,open('ipl_transformer.pkl','wb'))
print("Model Saved")